In [8]:

import pandas as pd
import numpy as np

from xgboost import XGBClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

df = pd.read_csv("creditcard.csv")

print(df.shape)
print(df["Class"].value_counts())
print("Missing values:", df.isnull().sum().sum())


(284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64
Missing values: 0


In [9]:

df = df.sort_values("Time")

X = df.drop(columns="Class")
y = df["Class"]

# 70% training, 15% validation, 15% test
n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
X_val = X.iloc[train_end:val_end]
X_test = X.iloc[val_end:]

y_train = y.iloc[:train_end]
y_val = y.iloc[train_end:val_end]
y_test = y.iloc[val_end:]

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


Training: (199364, 30)
Validation: (42721, 30)
Test: (42722, 30)


In [10]:

# Calculate class imbalance
fraud_count = (y_train == 1).sum()
genuine_count = (y_train == 0).sum()

imbalance_ratio = genuine_count / fraud_count

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    min_child_weight=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=imbalance_ratio * 0.5,
    random_state=42,
    n_jobs=-1,
    eval_metric="aucpr"
)

model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=-1, num_parallel_tree=None, ...)

In [11]:

prob = model.predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)

print("Average Precision:", average_precision_score(y_test, prob))
print("ROC-AUC:", roc_auc_score(y_test, prob))
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("F1:", f1_score(y_test, pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, pred))


Average Precision: 0.7691680429006749
ROC-AUC: 0.9804010203529772
Precision: 0.8260869565217391
Recall: 0.7307692307692307
F1: 0.7755102040816326
Confusion Matrix:
[[42662     8]
 [   14    38]]


In [12]:

# Indices of 100 highest fraud scores
top100 = np.argsort(prob)[-100:]

# Count actual frauds among those transactions
detected = np.asarray(y_test)[top100].sum()

total_frauds = y_test.sum()

print("Frauds detected:", detected)
print("False alarms:", 100 - detected)
print("Frauds missed:", total_frauds - detected)
print("Recall@100:", detected / total_frauds)
print("Precision@100:", detected / 100)


Frauds detected: 40
False alarms: 60
Frauds missed: 12
Recall@100: 0.7692307692307693
Precision@100: 0.4


In [13]:

import joblib

joblib.dump(model, "finauto_xgboost.pkl")
print("Model saved!")


Model saved!
